In [ ]:
pip install scikit-image

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Dataset
from transformers import ViTForImageClassification, ViTFeatureExtractor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from skimage.feature.texture import greycomatrix, greycoprops
from PIL import Image

In [25]:
# Define paths
train_metadata_path = "/kaggle/input/isic-data/ISIC_2019_Training_Metadata.csv"
train_ground_truth_path = "/kaggle/input/isic-data/ISIC_2019_Training_GroundTruth.csv"
test_metadata_path = "/kaggle/input/isic-data/ISIC_2019_Test_Metadata.csv"
test_ground_truth_path = "/kaggle/input/isic-data/ISIC_2019_Test_GroundTruth.csv"
train_image_dir = "/kaggle/input/isic-data/ISIC_2019_Training_Input/ISIC_2019_Training_Input"
test_image_dir = "/kaggle/input/isic-data/ISIC_2019_Test_Input/ISIC_2019_Test_Input"


In [26]:
def load_and_clean_metadata(metadata_path, ground_truth_path):
    metadata = pd.read_csv(metadata_path)
    ground_truth = pd.read_csv(ground_truth_path)

    # Handle missing values
    metadata.fillna({
        'age_approx': metadata['age_approx'].median(),  # Replace missing ages with median
        'anatom_site_general': 'unknown',             # Replace missing anatomical site with 'unknown'
        'sex': 'unknown'                              # Replace missing gender with 'unknown'
    }, inplace=True)

    # Merge metadata and ground truth on image ID
    combined_data = pd.merge(ground_truth, metadata, left_on='image', right_on='image')
    return combined_data


In [27]:
# Load and clean training and test datasets
train_data = load_and_clean_metadata(train_metadata_path, train_ground_truth_path)
test_data = load_and_clean_metadata(test_metadata_path, test_ground_truth_path)

In [28]:
train_data

,image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK,age_approx,anatom_site_general,lesion_id,sex
0,ISIC_0000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,55.0,anterior torso,NaN,female
1,ISIC_0000001,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,anterior torso,NaN,female
2,ISIC_0000002,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,upper extremity,NaN,female
3,ISIC_0000003,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,upper extremity,NaN,male
4,ISIC_0000004,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,80.0,posterior torso,NaN,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25326,ISIC_0073247,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,85.0,head/neck,BCN_0003925,female
25327,ISIC_0073248,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,65.0,anterior torso,BCN_0001819,male
25328,ISIC_0073249,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,70.0,lower extremity,BCN_0001085,male
25329,ISIC_0073251,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,55.0,palms/soles,BCN_0002083,female


In [29]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25331 entries, 0 to 25330
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   image                25331 non-null  object 
 1   MEL                  25331 non-null  float64
 2   NV                   25331 non-null  float64
 3   BCC                  25331 non-null  float64
 4   AK                   25331 non-null  float64
 5   BKL                  25331 non-null  float64
 6   DF                   25331 non-null  float64
 7   VASC                 25331 non-null  float64
 8   SCC                  25331 non-null  float64
 9   UNK                  25331 non-null  float64
 10  age_approx           25331 non-null  float64
 11  anatom_site_general  25331 non-null  object 
 12  lesion_id            23247 non-null  object 
 13  sex                  25331 non-null  object 
dtypes: float64(10), object(4)
memory usage: 2.7+ MB


In [30]:
test_data

,image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK,score_weight,validation_weight,age_approx,anatom_site_general,sex
0,ISIC_0034321,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,60.0,unknown,female
1,ISIC_0034322,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,70.0,anterior torso,male
2,ISIC_0034323,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,70.0,lower extremity,male
3,ISIC_0034324,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,70.0,lower extremity,male
4,ISIC_0034325,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,30.0,upper extremity,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8233,ISIC_0073236,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,75.0,anterior torso,male
8234,ISIC_0073243,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,65.0,lower extremity,male
8235,ISIC_0073250,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,30.0,anterior torso,female
8236,ISIC_0073252,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,85.0,head/neck,female


In [31]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8238 entries, 0 to 8237
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   image                8238 non-null   object 
 1   MEL                  8238 non-null   float64
 2   NV                   8238 non-null   float64
 3   BCC                  8238 non-null   float64
 4   AK                   8238 non-null   float64
 5   BKL                  8238 non-null   float64
 6   DF                   8238 non-null   float64
 7   VASC                 8238 non-null   float64
 8   SCC                  8238 non-null   float64
 9   UNK                  8238 non-null   float64
 10  score_weight         8238 non-null   float64
 11  validation_weight    8238 non-null   float64
 12  age_approx           8238 non-null   float64
 13  anatom_site_general  8238 non-null   object 
 14  sex                  8238 non-null   object 
dtypes: float64(12), object(3)
memory usage

In [37]:
# Add image path to check if images exist
train_data["image_path"] = train_data["image"].apply(lambda x: os.path.join(train_image_dir, x + ".jpg"))
test_data["image_path"] = test_data["image"].apply(lambda x: os.path.join(test_image_dir, x + ".jpg"))

# Check if all image paths exist
train_data["exists"] = train_data["image_path"].apply(lambda x: os.path.exists(x))
test_data["exists"] = test_data["image_path"].apply(lambda x: os.path.exists(x))



In [38]:
# Print the count of True/False for image existence
print("Training Data Image Existence Check:")
train_data["exists"].value_counts()  # Should be all True if paths are correct


Training Data Image Existence Check:


exists
True    25331
Name: count, dtype: int64

In [39]:
print("\nTesting Data Image Existence Check:")
test_data["exists"].value_counts()  # Should be all True if paths are correct


Testing Data Image Existence Check:


exists
True    8238
Name: count, dtype: int64